In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/healthcare.db")
print("✓ Connected to healthcare.db")

print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
).to_string(index=False))

✓ Connected to healthcare.db
    name
patients


In [2]:
# ── Query 1: Business Summary KPIs ──────────────────────────

query = """
SELECT
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted_30_day,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital,
    ROUND(AVG(age_numeric), 1)                  AS avg_patient_age,
    ROUND(AVG(num_medications), 1)              AS avg_medications,
    ROUND(AVG(number_diagnoses), 1)             AS avg_diagnoses
FROM patients
"""
pd.read_sql(query, conn)

,total_patients,readmitted_30_day,readmission_rate_pct,avg_days_in_hospital,avg_patient_age,avg_medications,avg_diagnoses
0,71518,6293,8.8,4.3,65.7,15.7,7.2


In [3]:
# ── Query 2: Readmission by Age Group ───────────────────────
# Do older patients get readmitted more?

query = """
SELECT
    age,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital
FROM patients
GROUP BY age
ORDER BY age
"""
pd.read_sql(query, conn)

,age,total_patients,readmitted,readmission_rate_pct,avg_days_in_hospital
0,[0-10),154,3,1.9,2.6
1,[10-20),535,26,4.9,2.9
2,[20-30),1127,83,7.4,3.5
3,[30-40),2699,188,7.0,3.6
4,[40-50),6878,507,7.4,3.9
5,[50-60),12466,879,7.1,4.0
6,[60-70),15960,1414,8.9,4.3
7,[70-80),18210,1824,10.0,4.5
8,[80-90),11589,1201,10.4,4.8
9,[90-100),1900,168,8.8,4.8


In [4]:
# ── Query 3: Readmission by Race ─────────────────────────────

query = """
SELECT
    race,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital
FROM patients
GROUP BY race
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql(query, conn)

,race,total_patients,readmitted,readmission_rate_pct,avg_days_in_hospital
0,Caucasian,53491,4816,9.0,4.3
1,AfricanAmerican,12887,1093,8.5,4.4
2,Asian,497,41,8.2,3.7
3,Hispanic,1517,122,8.0,4.0
4,Unknown,1948,141,7.2,4.2
5,Other,1178,80,6.8,4.2


In [5]:
# ── Query 4: Readmission by Time in Hospital ─────────────────
# Does longer stay mean higher readmission risk?

query = """
SELECT
    time_in_hospital,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(num_medications), 1)              AS avg_medications
FROM patients
GROUP BY time_in_hospital
ORDER BY time_in_hospital
"""
pd.read_sql(query, conn)

,time_in_hospital,total_patients,readmitted,readmission_rate_pct,avg_medications
0,1,10717,686,6.4,10.9
1,2,12397,937,7.6,12.2
2,3,12701,1025,8.1,14.1
3,4,9567,836,8.7,15.7
4,5,6839,655,9.6,17.3
5,6,5171,551,10.7,18.4
6,7,3999,448,11.2,19.8
7,8,2919,350,12.0,21.2
8,9,1990,236,11.9,22.3
9,10,1558,185,11.9,23.5


In [6]:
# ── Query 5: Readmission by Insulin Usage ────────────────────
# Insulin is the most important medication in diabetes management

query = """
SELECT
    insulin,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital,
    ROUND(AVG(num_medications), 1)              AS avg_medications
FROM patients
GROUP BY insulin
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql(query, conn)

,insulin,total_patients,readmitted,readmission_rate_pct,avg_days_in_hospital,avg_medications
0,Down,7505,775,10.3,4.8,18.9
1,Up,6963,671,9.6,5.3,19.7
2,Steady,22129,2005,9.1,4.3,16.1
3,No,34921,2842,8.1,4.0,14.0


In [7]:
# ── Query 6: High Risk Patient Profile (CTE) ─────────────────
# What does a typical high risk patient look like?

query = """
WITH high_risk AS (
    SELECT *
    FROM patients
    WHERE readmitted_30 = 1
),
low_risk AS (
    SELECT *
    FROM patients
    WHERE readmitted_30 = 0
)
SELECT
    'High Risk (Readmitted)'         AS patient_group,
    COUNT(*)                         AS total_patients,
    ROUND(AVG(age_numeric), 1)       AS avg_age,
    ROUND(AVG(time_in_hospital), 1)  AS avg_days_in_hospital,
    ROUND(AVG(num_medications), 1)   AS avg_medications,
    ROUND(AVG(number_diagnoses), 1)  AS avg_diagnoses,
    ROUND(AVG(n_meds_changed), 1)    AS avg_meds_changed,
    ROUND(AVG(service_utilization),1) AS avg_service_utilization
FROM high_risk
UNION ALL
SELECT
    'Low Risk (Not Readmitted)'      AS patient_group,
    COUNT(*)                         AS total_patients,
    ROUND(AVG(age_numeric), 1)       AS avg_age,
    ROUND(AVG(time_in_hospital), 1)  AS avg_days_in_hospital,
    ROUND(AVG(num_medications), 1)   AS avg_medications,
    ROUND(AVG(number_diagnoses), 1)  AS avg_diagnoses,
    ROUND(AVG(n_meds_changed), 1)    AS avg_meds_changed,
    ROUND(AVG(service_utilization),1) AS avg_service_utilization
FROM low_risk
"""
pd.read_sql(query, conn)

,patient_group,total_patients,avg_age,avg_days_in_hospital,avg_medications,avg_diagnoses,avg_meds_changed,avg_service_utilization
0,High Risk (Readmitted),6293,67.8,4.8,16.6,7.5,0.3,0.8
1,Low Risk (Not Readmitted),65225,65.4,4.2,15.6,7.2,0.3,0.5


In [8]:
# ── Query 7: Readmission by Number of Diagnoses ──────────────
# More diagnoses = higher complexity = higher readmission risk?

query = """
SELECT
    number_diagnoses,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital
FROM patients
GROUP BY number_diagnoses
ORDER BY number_diagnoses
"""
pd.read_sql(query, conn)

,number_diagnoses,total_patients,readmitted,readmission_rate_pct,avg_days_in_hospital
0,1,193,6,3.1,2.3
1,2,881,41,4.7,2.6
2,3,2364,140,5.9,2.8
3,4,4464,301,6.7,3.1
4,5,8941,685,7.7,3.7
5,6,7691,653,8.5,3.8
6,7,7609,646,8.5,4.0
7,8,7560,724,9.6,4.3
8,9,31740,3088,9.7,5.0
9,10,9,1,11.1,3.0


In [9]:
# ── Query 8: Patient Risk Ranking (Window Function) ──────────
# Ranks patients by readmission risk factors
# Shows window function skills

query = """
WITH patient_risk AS (
    SELECT
        encounter_id,
        age,
        time_in_hospital,
        num_medications,
        service_utilization,
        number_diagnoses,
        readmitted_30,
        ROUND(
            (time_in_hospital * 0.3) +
            (num_medications  * 0.2) +
            (service_utilization * 0.3) +
            (number_diagnoses * 0.2)
        , 1)                                    AS risk_score
    FROM patients
)
SELECT
    encounter_id,
    age,
    risk_score,
    readmitted_30,
    RANK() OVER (
        ORDER BY risk_score DESC
    )                                           AS overall_risk_rank,
    NTILE(4) OVER (
        ORDER BY risk_score DESC
    )                                           AS risk_quartile
FROM patient_risk
ORDER BY risk_score DESC
LIMIT 20
"""
pd.read_sql(query, conn)

,encounter_id,age,risk_score,readmitted_30,overall_risk_rank,risk_quartile
0,137878980,[50-60),21.0,0,1,1
1,161465688,[70-80),21.0,1,1,1
2,135412542,[60-70),20.4,1,3,1
3,167119920,[60-70),20.1,0,4,1
4,180371094,[70-80),20.1,0,4,1
5,443835140,[70-80),20.1,0,4,1
6,164471886,[50-60),19.9,0,7,1
7,123356280,[70-80),19.6,0,8,1
8,144470340,[60-70),19.6,0,8,1
9,229308276,[70-80),19.6,0,8,1


In [10]:
# ── Query 9: Readmission by A1C Result ───────────────────────
# A1C is a key diabetes management indicator
# High A1C = poor glucose control = higher readmission risk

query = """
SELECT
    A1Cresult,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital
FROM patients
GROUP BY A1Cresult
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql(query, conn)

,A1Cresult,total_patients,readmitted,readmission_rate_pct,avg_days_in_hospital
0,None,58532,5213,8.9,4.2
1,Norm,3791,324,8.5,4.9
2,>7,2891,246,8.5,4.8
3,>8,6304,510,8.1,4.7


In [11]:
# ── Query 10: Readmission by Admission Type ──────────────────
# Emergency admissions likely have higher readmission rates

query = """
SELECT
    CASE admission_type_id
        WHEN 1 THEN 'Emergency'
        WHEN 2 THEN 'Urgent'
        WHEN 3 THEN 'Elective'
        WHEN 4 THEN 'Newborn'
        WHEN 5 THEN 'Not Available'
        WHEN 6 THEN 'NULL'
        WHEN 7 THEN 'Trauma Center'
        WHEN 8 THEN 'Not Mapped'
        ELSE 'Other'
    END                                         AS admission_type,
    COUNT(*)                                    AS total_patients,
    SUM(readmitted_30)                          AS readmitted,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                        AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)             AS avg_days_in_hospital
FROM patients
GROUP BY admission_type_id
ORDER BY readmission_rate_pct DESC
"""
pd.read_sql(query, conn)

,admission_type,total_patients,readmitted,readmission_rate_pct,avg_days_in_hospital
0,Newborn,9,1,11.1,3.2
1,NULL,4588,452,9.9,4.5
2,Emergency,36490,3262,8.9,4.3
3,Urgent,13028,1149,8.8,4.5
4,Not Available,3174,265,8.3,3.9
5,Elective,13917,1143,8.2,4.1
6,Not Mapped,291,21,7.2,3.0
7,Trauma Center,21,0,0.0,4.9


In [12]:
sql_queries = """
-- ============================================================
-- HEALTHCARE PATIENT READMISSION ANALYSIS — SQL Queries
-- Database: data/healthcare.db | Table: patients
-- ============================================================

-- 1. BUSINESS SUMMARY
SELECT
    COUNT(*)                                AS total_patients,
    SUM(readmitted_30)                      AS readmitted_30_day,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                    AS readmission_rate_pct,
    ROUND(AVG(time_in_hospital), 1)         AS avg_days_in_hospital
FROM patients;

-- 2. READMISSION BY AGE GROUP
SELECT
    age,
    COUNT(*)                                AS total_patients,
    ROUND(SUM(readmitted_30) * 100.0
          / COUNT(*), 1)                    AS readmission_rate_pct
FROM patients
GROUP BY age
ORDER BY age;

-- 3. HIGH RISK PATIENT PROFILE (CTE + UNION)
WITH high_risk AS (
    SELECT * FROM patients WHERE readmitted_30 = 1
),
low_risk AS (
    SELECT * FROM patients WHERE readmitted_30 = 0
)
SELECT
    'High Risk' AS patient_group,
    ROUND(AVG(age_numeric), 1) AS avg_age,
    ROUND(AVG(time_in_hospital), 1) AS avg_days,
    ROUND(AVG(num_medications), 1) AS avg_medications
FROM high_risk
UNION ALL
SELECT
    'Low Risk',
    ROUND(AVG(age_numeric), 1),
    ROUND(AVG(time_in_hospital), 1),
    ROUND(AVG(num_medications), 1)
FROM low_risk;

-- 4. PATIENT RISK RANKING (WINDOW FUNCTION)
WITH patient_risk AS (
    SELECT encounter_id, age,
           ROUND((time_in_hospital * 0.3) +
                 (num_medications  * 0.2) +
                 (service_utilization * 0.3) +
                 (number_diagnoses * 0.2), 1) AS risk_score,
           readmitted_30
    FROM patients
)
SELECT
    encounter_id, age, risk_score, readmitted_30,
    RANK() OVER (ORDER BY risk_score DESC) AS risk_rank,
    NTILE(4) OVER (ORDER BY risk_score DESC) AS risk_quartile
FROM patient_risk
ORDER BY risk_score DESC
LIMIT 20;
"""

with open("../sql/healthcare_queries.sql", "w") as f:
    f.write(sql_queries)

conn.close()
print("✓ SQL file saved to sql/healthcare_queries.sql")
print("✓ Connection closed")

✓ SQL file saved to sql/healthcare_queries.sql
✓ Connection closed
